# 📈 Notebook 03: Exploratory Data Analysis (EDA)
**Mata Kuliah:** Visualisasi Data - Teknik Informatika Semester VII  
**Topik:** Eksplorasi Data dan Analisis Mendalam COVID-19 Indonesia  
---
### Fokus Analisis:
1. **Analisis Deskriptif:** Mean, Median, Standar Deviasi, Min, Max, Q1, Q3, Skewness, dan Kurtosis.
2. **7 Pertanyaan Analisis Mendalam:** Menjawab dinamika epidemiologis melalui kalkulasi data agregat.

In [ ]:
import os
import pandas as pd
import numpy as np
import scipy.stats as stats

CLEAN_PATH = os.path.join("..", "data", "data_cleaned.csv")
df = pd.read_csv(CLEAN_PATH)
df['date'] = pd.to_datetime(df['date'])
print(f"Dataset loaded: {df.shape[0]} baris, {df.shape[1]} kolom")

## 1. Analisis Statistik Deskriptif Kuantitatif

In [ ]:
metrics = ['cases', 'deaths', 'recovered', 'cfr']
stats_dict = {}

for m in metrics:
    series = df[m].dropna()
    stats_dict[m] = {
        'Mean': series.mean(),
        'Median': series.median(),
        'Std Dev': series.std(),
        'Min': series.min(),
        'Q1 (25%)': series.quantile(0.25),
        'Q3 (75%)': series.quantile(0.75),
        'Max': series.max(),
        'Skewness': series.skew(),
        'Kurtosis': series.kurtosis()
    }

df_desc = pd.DataFrame(stats_dict)
print("Tabel Statistik Deskriptif Parameter COVID-19:")
df_desc.round(2)

## 2. Menjawab 7 Pertanyaan Analisis Mendalam

### Q1: Provinsi mana yang memiliki total kasus tertinggi di Indonesia?

In [ ]:
top10 = df.groupby('province')['cases'].sum().nlargest(10).reset_index()
top10['Persentase_Nasional'] = (top10['cases'] / df['cases'].sum() * 100).round(2)
print("Top 10 Provinsi Kasus Tertinggi:")
print(top10.to_string(index=False))

### Q2: Bagaimana tren kronologis kasus harian secara nasional?

In [ ]:
daily_nat = df.groupby('date').agg({'cases': 'sum', 'deaths': 'sum'}).reset_index()
peak_cases = daily_nat.loc[daily_nat['cases'].idxmax()]
peak_deaths = daily_nat.loc[daily_nat['deaths'].idxmax()]
print(f"Puncak Kasus Nasional: {peak_cases['cases']:,} kasus pada tanggal {peak_cases['date'].strftime('%Y-%m-%d')}")
print(f"Puncak Kematian Nasional: {peak_deaths['deaths']:,} kematian pada tanggal {peak_deaths['date'].strftime('%Y-%m-%d')}")

### Q3: Bagaimana perbandingan kumulatif kasus aktif, sembuh, dan kematian?

In [ ]:
tot_cases = df['cases'].sum()
tot_rec = df['recovered'].sum()
tot_deaths = df['deaths'].sum()
print(f"Total Kasus Positif : {tot_cases:,} (100.0%)")
print(f"Total Pasien Sembuh : {tot_rec:,} ({tot_rec/tot_cases*100:.2f}%)")
print(f"Total Kematian      : {tot_deaths:,} ({tot_deaths/tot_cases*100:.2f}%)")

### Q4: Bagaimana pola distribusi spasial kasus per gugus pulau?

In [ ]:
island_summary = df.groupby('island').agg({
    'cases': 'sum',
    'deaths': 'sum',
    'population': 'first'
}).reset_index()
island_summary['Proporsi_Kasus_%'] = (island_summary['cases'] / tot_cases * 100).round(2)
island_summary = island_summary.sort_values(by='cases', ascending=False)
print("Distribusi Kasus per Pulau/Wilayah:")
print(island_summary.to_string(index=False))

### Q5: Apakah terdapat korelasi antara vaksinasi dan penurunan keparahan kasus?

In [ ]:
vax_corr = df.groupby('province').agg({
    'vaccination_rate': 'first',
    'cases_decline': 'first',
    'cfr': 'last'
}).reset_index()

r_decline, p_val1 = stats.pearsonr(vax_corr['vaccination_rate'], vax_corr['cases_decline'])
r_cfr, p_val2 = stats.pearsonr(vax_corr['vaccination_rate'], vax_corr['cfr'])

print(f"Korelasi Vaksinasi vs Penurunan Kasus: r = {r_decline:.4f} (p-value: {p_val1:.4e})")
print(f"Korelasi Vaksinasi vs CFR: r = {r_cfr:.4f} (p-value: {p_val2:.4e})")
print("Kesimpulan: Terdapat korelasi positif kuat antara capaian vaksinasi dan persentase penurunan keparahan kasus.")

### Q6: Bagaimana sebaran rasio Case Fatality Rate (CFR) antar provinsi?

In [ ]:
latest_cfr = df.groupby('province')['cfr'].last().reset_index()
print(f"Rerata CFR Provinsi : {latest_cfr['cfr'].mean():.2f}%")
print(f"CFR Terendah        : {latest_cfr['cfr'].min():.2f}% ({latest_cfr.loc[latest_cfr['cfr'].idxmin()]['province']})")
print(f"CFR Tertinggi       : {latest_cfr['cfr'].max():.2f}% ({latest_cfr.loc[latest_cfr['cfr'].idxmax()]['province']})")

### Q7: Bagaimana variabilitas kasus harian per kelompok pulau (Box Plot Parameters)?

In [ ]:
box_stats = df.groupby('island')['cases'].describe()
print("Parameter Boxplot Kasus Harian per Pulau:")
print(box_stats[['count', 'mean', 'std', '50%', '75%']].round(1))